<a href="https://colab.research.google.com/github/replysantosh-lang/ECARAgenticAI/blob/main/Lab3_CrewAIManufacturingSupportTriageSystem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CrewAI Manufacturing Support Triage System

A four-agent sequential CrewAI pipeline that triages, diagnoses, analyses, and contains a manufacturing shop floor support ticket.
Demonstrates agent role definition, task context chaining, and sequential process execution.

Requires: pip install crewai | OpenAI API key in Colab secrets. Use Gemini to fix any CrewAI installation version conflicts.

In [1]:
!pip install crewai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 10.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 8.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.9/185.9 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from crewai import Agent, Task, Crew, Process
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

support_ticket = """
Ticket ID: PLT7-PKG3-001 | Plant 7, Packaging Line 3
Issue: Carton sealer intermittently jams after speed increases to 120 units/min.
Impact: Line stopped 4 times in last 2 hours. 1,800 units at risk of delay.
Signals: High reject count, operator reports adhesive overflow.
Recent change: Changeover from SKU A to SKU C.
"""

triage_agent = Agent(
    role="Manufacturing Support Triage Agent",
    goal="Classify the shop floor issue and identify immediate containment actions.",
    backstory="You support high-volume manufacturing lines and understand downtime escalation protocols.",
    verbose=True
)
maintenance_agent = Agent(
    role="Maintenance Troubleshooting Agent",
    goal="Diagnose equipment-related causes and recommend safe maintenance checks.",
    backstory="You are a maintenance engineer familiar with packaging equipment and line stoppages.",
    verbose=True
)
process_agent = Agent(
    role="Process Engineering Agent",
    goal="Assess whether process parameters, SKU changeover, or line settings caused the issue.",
    backstory="You optimise manufacturing process parameters and changeover standards.",
    verbose=True
)
quality_agent = Agent(
    role="Quality Containment Agent",
    goal="Define quarantine criteria, inspection scope, and restart conditions.",
    backstory="You ensure quality standards during and after line interruptions.",
    verbose=True
)

t1 = Task(description=f"Triage this manufacturing support ticket:\n{support_ticket}",
          expected_output="Priority classification, missing information list, immediate actions.",
          agent=triage_agent)
t2 = Task(description="Diagnose likely equipment causes and propose maintenance checks.",
          expected_output="Root cause hypothesis and 3-5 maintenance actions.",
          agent=maintenance_agent, context=[t1])
t3 = Task(description="Analyse whether process parameters or changeover settings caused the issue.",
          expected_output="Process parameter findings and recommended adjustments.",
          agent=process_agent, context=[t1, t2])
t4 = Task(description="Define quality containment and restart criteria.",
          expected_output="Quarantine scope, inspection plan, and restart sign-off criteria.",
          agent=quality_agent, context=[t1, t2, t3])

crew = Crew(agents=[triage_agent, maintenance_agent, process_agent, quality_agent],
            tasks=[t1, t2, t3], process=Process.sequential, verbose=True)
result = await crew.kickoff_async()
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: df77d167-5c99-4530-a5ab-30b36e840e57                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Triage this manufacturing support ticket:                                                                │
│                                                                                                                 │
│  Ticket ID: PLT7-PKG3-001 | Plant 7, Packaging Line 3                                                           │
│  Issue: Carton sealer intermittently jams after speed increases to 120 units/min.                               │
│  Impact: Line stopped 4 times in last 2 hours. 1,800 units at risk of delay.                                    │
│  Signals: High reject count, operator reports adhesive overflow.                                                │
│  Recent change: Changeover from SKU A to SKU C.                                                                 │
│                                                                                                                 │
│  ID: fb34bdcb-20a4-4cad-bb03-3034c8cabfbd                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Manufacturing Support Triage Agent                                                                      │
│                                                                                                                 │
│  Task: Triage this manufacturing support ticket:                                                                │
│                                                                                                                 │
│  Ticket ID: PLT7-PKG3-001 | Plant 7, Packaging Line 3                                                           │
│  Issue: Carton sealer intermittently jams after speed increases to 120 units/min.                               │
│  Impact: Line stopped 4 times in last 2 hours. 1,800 units at risk of delay.                                    │
│  Signals: High reject count, operator reports adhesive overflow.                                                │
│  Recent change: Changeover from SKU A to SKU C.                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Manufacturing Support Triage Agent                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Priority Classification:                                                                                       │
│  High Priority - The intermittent jamming of the carton sealer at increased speeds is causing frequent line     │
│  stoppages (4 times in 2 hours) and risks delaying 1,800 units, which significantly impacts production output   │
│  and delivery commitments.                                                                                      │
│                                                                                                                 │
│  Missing Information List:                                                                                      │
│  1. Has the adhesive application system been checked for correct settings and maintenance after the SKU         │
│  change?                                                                                                        │
│  2. Are there recorded differences in carton specifications or tolerances between SKU A and SKU C?              │
│  3. What is the current setting of the carton sealer's speed and adhesive parameters?                           │
│  4. Are there any alarms or error codes generated by the carton sealer during the jams?                         │
│  5. Has the sealing temperature and pressure been verified and adjusted for SKU C?                              │
│  6. Is the operator following the standard operating procedure (SOP) for changeover to SKU C?                   │
│  7. Are there any visible mechanical issues like worn parts or misalignments in the sealer?                     │
│  8. Confirm the amount and nature of adhesive overflow reported – is it excessive or inconsistent with normal   │
│  operation?                                                                                                     │
│                                                                                                                 │
│  Immediate Actions:                                                                                             │
│  1. Immediately reduce the carton sealer speed to the previous stable rate (<120 units/min) to prevent further  │
│  jams and line stoppages.                                                                                       │
│  2. Instruct operators to contain and clean adhesive overflow promptly to avoid further carton jams and         │
│  maintain product quality.                                                                                      │
│  3. Perform a quick visual and functional inspection of the carton sealer, focusing on the adhesive             │
│  application system, sealing temperature, pressure, and mechanical components.                                  │
│  4. Review and compare the adhesive and carton specifications between SKU A and SKU C to identify differences   │
│  that could affect sealing.                                                                                     │
│  5. Notify the maintenance team to prepare for a detailed inspection and adjustment to adhesive application     │
│  settings in line with SKU C requirements.                                                                      │
│  6. Verify that changeover procedures were correctly followed and provide refresher training if necessary.      │
│  7. Document each jam event with detailed notes, includ

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Process Engineering Agent                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Process Parameter Findings and Recommended Adjustments:                                                        │
│                                                                                                                 │
│  1. Adhesive Application Parameters:                                                                            │
│  - The adhesive application system for SKU C requires reassessment as current pump rates, nozzle conditions,    │
│  and spray patterns appear to be misaligned with the new carton specifications. Excessive adhesive overflow     │
│  indicates that adhesive volume and timing have not been suitably adjusted post changeover.                     │
│  - It is necessary to verify adhesive type compatibility with SKU C’s carton material; viscosity and            │
│  temperature settings must be calibrated within the recommended specification range to ensure consistent flow   │
│  and adhesion.                                                                                                  │
│  Recommended Adjustment:                                                                                        │
│  - Reduce adhesive pump rate and optimize spray pattern nozzle condition to achieve uniform adhesive spread     │
│  without overflow.                                                                                              │
│  - Adjust adhesive temperature and monitor viscosity to align with SKU C requirements.                          │
│  - Implement fine-tuning of adhesive timing to apply adhesive only when cartons are correctly positioned,       │
│  minimizing waste and sticking.                                                                                 │
│                                                                                                                 │
│  2. Sealing Temperature and Pressure Settings:                                                                  │
│  - Current settings for sealing temperature and pressure are not verified for SKU C material and thickness.     │
│  Improper sealing pressure and temperature can cause insufficient bonding or excess adhesive spread leading to  │
│  jams.                                                                                                          │
│  Recommended Adjustment:                                                                                        │
│  - Calibrate sealing temperature to match SKU C carton’s thermal properties.                                    │
│  - Adjust sealing pressure to provide adequate bonding without deforming cartons.                               │
│  - Replace any worn heaters or pressure rollers impacting effectiveness.                                        │
│                                                                                                                 │
│  3. Carton Sealer Speed:                                                                                        │
│  - Increasing speed beyond the previously stable rate (<120 units/min) has directly correlated with             │
│  intermittent jams primarily due to insufficient adhesive curing time and increased mechanical stress on        │
│  imperfect sealing.                                                                                             │
│  Recommended Adjustment:                               

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Analyse whether process parameters or changeover settings caused the issue.                              │
│  Agent: Process Engineering Agent                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Process Parameter Findings and Recommended Adjustments:

1. Adhesive Application Parameters:
- The adhesive application system for SKU C requires reassessment as current pump rates, nozzle conditions, and spray patterns appear to be misaligned with the new carton specifications. Excessive adhesive overflow indicates that adhesive volume and timing have not been suitably adjusted post changeover.
- It is necessary to verify adhesive type compatibility with SKU C’s carton material; viscosity and temperature settings must be calibrated within the recommended specification range to ensure consistent flow and adhesion.
Recommended Adjustment:
- Reduce adhesive pump rate and optimize spray pattern nozzle condition to achieve uniform adhesive spread without overflow.
- Adjust adhesive temperature and monitor viscosity to align with SKU C requirements.
- Implement fine-tuning of adhesive timing to apply adhesive only when cartons are correctly positioned, minimizing waste and sticking.

2. Sea

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: df77d167-5c99-4530-a5ab-30b36e840e57                                                                       │
│  Final Output: Process Parameter Findings and Recommended Adjustments:                                          │
│                                                                                                                 │
│  1. Adhesive Application Parameters:                                                                            │
│  - The adhesive application system for SKU C requires reassessment as current pump rates, nozzle conditions,    │
│  and spray patterns appear to be misaligned with the new carton specifications. Excessive adhesive overflow     │
│  indicates that adhesive volume and timing have not been suitably adjusted post changeover.                     │
│  - It is necessary to verify adhesive type compatibility with SKU C’s carton material; viscosity and            │
│  temperature settings must be calibrated within the recommended specification range to ensure consistent flow   │
│  and adhesion.                                                                                                  │
│  Recommended Adjustment:                                                                                        │
│  - Reduce adhesive pump rate and optimize spray pattern nozzle condition to achieve uniform adhesive spread     │
│  without overflow.                                                                                              │
│  - Adjust adhesive temperature and monitor viscosity to align with SKU C requirements.                          │
│  - Implement fine-tuning of adhesive timing to apply adhesive only when cartons are correctly positioned,       │
│  minimizing waste and sticking.                                                                                 │
│                                                                                                                 │
│  2. Sealing Temperature and Pressure Settings:                                                                  │
│  - Current settings for sealing temperature and pressure are not verified for SKU C material and thickness.     │
│  Improper sealing pressure and temperature can cause insufficient bonding or excess adhesive spread leading to  │
│  jams.                                                                                                          │
│  Recommended Adjustment:                                                                                        │
│  - Calibrate sealing temperature to match SKU C carton’s thermal properties.                                    │
│  - Adjust sealing pressure to provide adequate bonding without deforming cartons.                               │
│  - Replace any worn heaters or pressure rollers impacting effectiveness.                                        │
│                                                                                                                 │
│  3. Carton Sealer Speed:                                                                                        │
│  - Increasing speed beyond the previously stable rate (<120 units/min) has directly correlated with             │
│  intermittent jams primarily due to insufficient adhesive curing time and increased mechanical stress on        │
│  imperfect sealing.                                                                                             │
│  Recommended Adjustment:                              



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

# ⚠ The crew runs async (kickoff_async) so subtasks within each stage can proceed without blocking. The top-level task sequence remains strictly sequential: T2 does not start until T1 is complete. Add a judge agent as T5 to score the quality of the final resolution.